# Prepare the complete Xenium RCC dataset for `reach-gap`

This notebook processes the **full 36.1 GB Xenium output bundle** directly from Google Drive without loading the whole archive into RAM or extracting every member.

It will:

- verify the four downloaded files against the provider sizes and MD5 checksums;
- inventory the full ZIP;
- selectively extract the cell table, full cell-feature HDF5 matrix, panels, metrics and small secondary-analysis files;
- process every detected cell and every matrix feature needed for vessel, CAF/ECM, tumour-neighbourhood and target scoring;
- use Xenium protein intensity when available for PD-L1, VISTA, PD-1 and LAG-3;
- calculate nearest endothelial-cell distance and local barrier scores;
- attempt to align the supplied pathology polygons, but abstain if the transform is ambiguous;
- write resumable, partitioned outputs to Drive.

**Scientific boundary:** Xenium protein values are scaled mean fluorescence intensities, not surface-antigen molecules per cell. Therefore the absolute mechanistic reachability index remains `NOT_COMPUTED` until an independent calibration is supplied. The preparation itself is real-data analysis, not a simulation.

In [ ]:
# Configuration
# False is the practical default: exact sizes and all files below 1 GB are MD5-checked.
# Set True for a full provider-MD5 pass over ~40 GB before processing.
VERIFY_LARGE_MD5 = False
FORCE_RERUN = False       # Leave False to resume from checkpoints after a Colab interruption.
RAW_DIR_HINT = None       # Optional exact mounted path if automatic discovery finds 0 or >1 folder.
RANDOM_SEED = 17

EXPECTED_FILES = [
    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_outs.zip",
    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_image.ome.tif",
    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_imagealignment.csv",
    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_annotation.geojson",
]


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
%pip install -q "h5py>=3.10" "pyarrow>=15.0" "shapely>=2.0" "tifffile>=2024.1"


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import sys

import psutil

print("Python:", platform.python_version())
print("CPU cores:", os.cpu_count())
print(f"RAM total: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"RAM available: {psutil.virtual_memory().available / 1024**3:.1f} GB")
print(f"Local disk free: {shutil.disk_usage('/content').free / 1024**3:.1f} GB")


def locate_raw_dir() -> Path:
    if RAW_DIR_HINT:
        candidate = Path(RAW_DIR_HINT)
        missing = [name for name in EXPECTED_FILES if not (candidate / name).exists()]
        if missing:
            raise FileNotFoundError(f"RAW_DIR_HINT is missing files: {missing}")
        return candidate

    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    zip_name = EXPECTED_FILES[0]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob(zip_name):
            candidate = zip_path.parent
            if all((candidate / name).exists() for name in EXPECTED_FILES):
                matches.append(candidate)
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise RuntimeError(
            "Could not identify exactly one raw-data folder. "
            f"Found {len(unique)} candidates: {unique}. Set RAW_DIR_HINT manually."
        )
    return unique[0]


RAW_DIR = locate_raw_dir()
OUTPUT_DIR = RAW_DIR / "reach-gap-analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Raw data:", RAW_DIR)
print("Outputs:", OUTPUT_DIR)


In [ ]:
from pathlib import Path

MODULE_SOURCE = '"""Low-memory preparation of Xenium gene-and-protein data for reach-gap.\n\nThis module intentionally prepares auditable cell-level inputs. It does not convert\nXenium protein intensity or RNA abundance into absolute surface antigen density.\nAny downstream mechanistic index must therefore abstain unless an independent\ncalibration is supplied.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport math\nimport shutil\nimport zipfile\nfrom collections.abc import Iterable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Literal\n\nimport numpy as np\nimport pandas as pd\nfrom numpy.typing import NDArray\nfrom scipy.spatial import cKDTree\n\nFloatArray = NDArray[np.float64]\n\n\n@dataclass(frozen=True)\nclass XeniumFeatureCatalog:\n    """Feature metadata and matrix dimensions from a 10x HDF5 matrix."""\n\n    feature_ids: tuple[str, ...]\n    feature_names: tuple[str, ...]\n    feature_types: tuple[str, ...]\n    barcodes: tuple[str, ...]\n    matrix_shape: tuple[int, int]\n\n\n@dataclass(frozen=True)\nclass AlignmentCandidate:\n    """One candidate mapping of annotation coordinates to Xenium microns."""\n\n    name: str\n    matrix: NDArray[np.float64]\n    scale_x: float\n    scale_y: float\n    score: float\n    fraction_inside: float\n\n\nDEFAULT_MARKERS: dict[str, tuple[str, ...]] = {\n    "endothelial": (\n        "CD31",\n        "PECAM1",\n        "VWF",\n        "EMCN",\n        "KDR",\n        "ENG",\n        "RAMP2",\n        "PLVAP",\n    ),\n    "pericyte_smooth_muscle": (\n        "alphaSMA",\n        "ACTA2",\n        "RGS5",\n        "CSPG4",\n        "MCAM",\n        "PDGFRB",\n        "DES",\n    ),\n    "caf": (\n        "alphaSMA",\n        "Vimentin",\n        "VIM",\n        "FAP",\n        "PDGFRA",\n        "PDGFRB",\n        "COL1A1",\n        "COL1A2",\n        "COL3A1",\n        "DCN",\n        "LUM",\n        "SPARC",\n        "FN1",\n    ),\n    "ecm": (\n        "COL1A1",\n        "COL1A2",\n        "COL3A1",\n        "COL4A1",\n        "COL4A2",\n        "COL6A1",\n        "COL6A2",\n        "FN1",\n        "LAMA4",\n        "LAMB1",\n        "DCN",\n        "LUM",\n        "SPARC",\n    ),\n    "epithelial_malignant": (\n        "PanCK",\n        "E-cadherin",\n        "Beta-catenin",\n        "EPCAM",\n        "KRT7",\n        "KRT8",\n        "KRT18",\n        "KRT19",\n        "CA9",\n        "PAX8",\n        "KIM1",\n        "HAVCR1",\n    ),\n    "immune": (\n        "CD45",\n        "PTPRC",\n        "CD3E",\n        "CD4",\n        "CD8A",\n        "CD20",\n        "MS4A1",\n        "CD68",\n        "CD163",\n        "CD11c",\n        "ITGAX",\n    ),\n}\n\nDEFAULT_TARGET_ALIASES: dict[str, tuple[str, ...]] = {\n    "PD-L1": ("PD-L1", "CD274"),\n    "VISTA": ("VISTA", "VSIR"),\n    "PD-1": ("PD-1", "PDCD1"),\n    "LAG-3": ("LAG-3", "LAG3"),\n}\n\nEXPECTED_RCC_FILES: dict[str, dict[str, Any]] = {\n    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_outs.zip": {\n        "size": 36_149_509_228,\n        "md5": "76d46bac8060f8bc3ecb450e03b4f3f6",\n    },\n    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_image.ome.tif": {\n        "size": 3_720_697_771,\n        "md5": "96ad5f699c7d6280cdf6af1c13f39515",\n    },\n    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_imagealignment.csv": {\n        "size": 129,\n        "md5": "e78bbee6561b9c037cd3eb839f63272",\n    },\n    "Xenium_V1_Human_Kidney_FFPE_Protein_updated_annotation.geojson": {\n        "size": 65_584,\n        "md5": "b5e848d7147f25817568d5871592eb56",\n    },\n}\n\n\ndef _decode(values: NDArray[Any]) -> tuple[str, ...]:\n    decoded: list[str] = []\n    for value in values:\n        if isinstance(value, bytes):\n            decoded.append(value.decode("utf-8"))\n        else:\n            decoded.append(str(value))\n    return tuple(decoded)\n\n\ndef md5_file(path: Path, *, chunk_size: int = 16 * 1024 * 1024) -> str:\n    """Compute a streaming MD5 for comparison with provider checksums."""\n\n    digest = hashlib.md5()  # noqa: S324 - provider publishes MD5 for transfer integrity.\n    with path.open("rb") as handle:\n        while chunk := handle.read(chunk_size):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef sha256_file(path: Path, *, chunk_size: int = 16 * 1024 * 1024) -> str:\n    """Compute a streaming SHA-256 for the run provenance manifest."""\n\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while chunk := handle.read(chunk_size):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_expected_files(\n    raw_dir: Path,\n    *,\n    verify_large_md5: bool = True,\n    expected: Mapping[str, Mapping[str, Any]] = EXPECTED_RCC_FILES,\n) -> dict[str, Any]:\n    """Validate names, sizes and provider MD5 values for the RCC download."""\n\n    report: dict[str, Any] = {\n        "files": {},\n        "all_present": True,\n        "all_verified": True,\n        "all_md5_computed": True,\n        "verification_policy": (\n            "size_and_provider_md5_for_all_files"\n            if verify_large_md5\n            else "size_for_large_files_and_provider_md5_below_1GB"\n        ),\n    }\n    for name, specification in expected.items():\n        path = raw_dir / name\n        entry: dict[str, Any] = {\n            "path": str(path),\n            "present": path.exists(),\n            "expected_size": int(specification["size"]),\n            "expected_md5": str(specification["md5"]),\n        }\n        if not path.exists():\n            report["all_present"] = False\n            report["all_verified"] = False\n            report["files"][name] = entry\n            continue\n        size = path.stat().st_size\n        entry["observed_size"] = size\n        entry["size_matches"] = size == int(specification["size"])\n        should_hash = verify_large_md5 or size < 1_000_000_000\n        if should_hash:\n            observed_md5 = md5_file(path)\n            entry["observed_md5"] = observed_md5\n            entry["md5_matches"] = observed_md5 == str(specification["md5"])\n        else:\n            entry["observed_md5"] = None\n            entry["md5_matches"] = None\n            entry["md5_status"] = "NOT_COMPUTED_BY_CONFIGURATION"\n            report["all_md5_computed"] = False\n        if not entry["size_matches"] or entry.get("md5_matches") is False:\n            report["all_verified"] = False\n        report["files"][name] = entry\n    return report\n\n\ndef inspect_zip(zip_path: Path) -> pd.DataFrame:\n    """Return an inventory of every member without extracting the archive."""\n\n    rows: list[dict[str, Any]] = []\n    with zipfile.ZipFile(zip_path) as archive:\n        for info in archive.infolist():\n            rows.append(\n                {\n                    "member": info.filename,\n                    "basename": Path(info.filename).name,\n                    "uncompressed_bytes": info.file_size,\n                    "compressed_bytes": info.compress_size,\n                    "compression": info.compress_type,\n                    "crc32": f"{info.CRC:08x}",\n                    "is_dir": info.is_dir(),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef _members_by_basename(inventory: pd.DataFrame) -> dict[str, list[str]]:\n    mapping: dict[str, list[str]] = {}\n    for basename, group in inventory.groupby("basename", sort=False):\n        mapping[str(basename)] = [str(value) for value in group["member"].tolist()]\n    return mapping\n\n\ndef select_essential_members(inventory: pd.DataFrame) -> list[str]:\n    """Select the minimum full-resolution files needed for cell-level preparation."""\n\n    by_name = _members_by_basename(inventory)\n    selected: list[str] = []\n    preferred_groups = (\n        ("cells.parquet", "cells.csv.gz"),\n        ("cell_feature_matrix.h5",),\n        ("metrics_summary.csv",),\n        ("gene_panel.json",),\n        ("protein_panel.json",),\n        ("experiment.xenium",),\n        ("analysis_summary.html",),\n        ("overview_scan.png",),\n    )\n    for alternatives in preferred_groups:\n        for basename in alternatives:\n            candidates = by_name.get(basename, [])\n            if candidates:\n                selected.append(candidates[0])\n                break\n    # Secondary analysis CSVs are small and useful for QC, but never pull image payloads.\n    for row in inventory.itertuples(index=False):\n        member = str(row.member)\n        if (\n            not bool(row.is_dir)\n            and member.lower().endswith(".csv")\n            and "/analysis/" in f"/{member.lower()}"\n            and int(row.uncompressed_bytes) <= 100_000_000\n        ):\n            selected.append(member)\n    return sorted(set(selected))\n\n\ndef extract_members(zip_path: Path, members: Sequence[str], output_dir: Path) -> dict[str, str]:\n    """Extract selected members while preserving their relative paths."""\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    extracted: dict[str, str] = {}\n    with zipfile.ZipFile(zip_path) as archive:\n        archive_names = set(archive.namelist())\n        for member in members:\n            if member not in archive_names:\n                raise KeyError(f"Archive member not found: {member}")\n            destination = output_dir / member\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(member) as source, destination.open("wb") as target:\n                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)\n            extracted[member] = str(destination)\n    return extracted\n\n\ndef find_extracted(output_dir: Path, basename: str) -> Path | None:\n    """Find one extracted file by basename, rejecting ambiguous duplicates."""\n\n    matches = [path for path in output_dir.rglob(basename) if path.is_file()]\n    if not matches:\n        return None\n    if len(matches) > 1:\n        raise ValueError(f"Multiple extracted files named {basename}: {matches}")\n    return matches[0]\n\n\ndef read_cells(path: Path) -> pd.DataFrame:\n    """Read and validate the Xenium cell summary."""\n\n    if path.suffix == ".parquet":\n        table = pd.read_parquet(path)\n    elif path.name.endswith(".csv.gz"):\n        table = pd.read_csv(path, compression="gzip")\n    else:\n        table = pd.read_csv(path)\n    required = {"cell_id", "x_centroid", "y_centroid", "cell_area", "nucleus_area"}\n    missing = required.difference(table.columns)\n    if missing:\n        raise ValueError(f"Xenium cells table is missing: {sorted(missing)}")\n    if table["cell_id"].duplicated().any():\n        raise ValueError("Xenium cells table contains duplicate cell_id values")\n    table = table.copy()\n    table["cell_id"] = table["cell_id"].astype(str)\n    table["x_um"] = pd.to_numeric(table["x_centroid"], errors="raise")\n    table["y_um"] = pd.to_numeric(table["y_centroid"], errors="raise")\n    if not np.isfinite(table[["x_um", "y_um"]].to_numpy(dtype=np.float64)).all():\n        raise ValueError("Cell coordinates contain non-finite values")\n    return table\n\n\ndef read_10x_h5_catalog(path: Path) -> XeniumFeatureCatalog:\n    """Read feature metadata from a standard 10x cell-feature HDF5 matrix."""\n\n    import h5py\n\n    with h5py.File(path, "r") as handle:\n        if "matrix" not in handle:\n            raise ValueError("HDF5 file has no /matrix group")\n        matrix = handle["matrix"]\n        features = matrix["features"]\n        names_key = "name" if "name" in features else "gene_names"\n        ids_key = "id" if "id" in features else "genes"\n        feature_ids = _decode(features[ids_key][...])\n        feature_names = _decode(features[names_key][...])\n        if "feature_type" in features:\n            feature_types = _decode(features["feature_type"][...])\n        else:\n            feature_types = tuple("Gene Expression" for _ in feature_names)\n        barcodes = _decode(matrix["barcodes"][...])\n        raw_shape = tuple(int(value) for value in matrix["shape"][...])\n    if len(raw_shape) != 2:\n        raise ValueError(f"Unexpected matrix shape: {raw_shape}")\n    if raw_shape != (len(feature_names), len(barcodes)):\n        raise ValueError(\n            "HDF5 shape does not match feature/barcode arrays: "\n            f"{raw_shape}, {len(feature_names)}, {len(barcodes)}"\n        )\n    return XeniumFeatureCatalog(\n        feature_ids=feature_ids,\n        feature_names=feature_names,\n        feature_types=feature_types,\n        barcodes=barcodes,\n        matrix_shape=raw_shape,\n    )\n\n\ndef validate_cell_barcode_identity(\n    cells: pd.DataFrame, catalog: XeniumFeatureCatalog\n) -> None:\n    """Require exact cell-identifier identity before joining matrix and geometry."""\n\n    if len(cells) != len(catalog.barcodes):\n        raise ValueError(\n            f"Cell table/HDF5 barcode count mismatch: {len(cells)} vs {len(catalog.barcodes)}"\n        )\n    cell_ids = set(cells["cell_id"].astype(str))\n    matrix_barcodes = set(catalog.barcodes)\n    if cell_ids != matrix_barcodes:\n        missing_from_cells = sorted(matrix_barcodes.difference(cell_ids))[:10]\n        missing_from_matrix = sorted(cell_ids.difference(matrix_barcodes))[:10]\n        raise ValueError(\n            "Cell table/HDF5 barcode identities differ; refusing to impute unmatched cells. "\n            f"Examples missing from cells: {missing_from_cells}; "\n            f"examples missing from matrix: {missing_from_matrix}"\n        )\n\n\ndef _is_protein_feature_type(feature_type: str) -> bool:\n    """Recognize Xenium/10x protein feature labels across software versions."""\n\n    normalized = _normalise_token(feature_type)\n    return "PROTEIN" in normalized or "ANTIBODY" in normalized\n\n\ndef canonical_feature_name(name: str, feature_type: str) -> str:\n    """Create a stable, collision-resistant column name."""\n\n    prefix = "protein" if _is_protein_feature_type(feature_type) else "rna"\n    cleaned = "".join(character if character.isalnum() else "_" for character in name)\n    cleaned = "_".join(part for part in cleaned.split("_") if part)\n    return f"{prefix}__{cleaned}"\n\n\ndef _normalise_token(value: str) -> str:\n    return "".join(character for character in value.upper() if character.isalnum())\n\n\ndef resolve_feature_indices(\n    catalog: XeniumFeatureCatalog,\n    *,\n    markers: Mapping[str, Sequence[str]] = DEFAULT_MARKERS,\n    targets: Mapping[str, Sequence[str]] = DEFAULT_TARGET_ALIASES,\n    include_all_proteins: bool = True,\n) -> tuple[list[int], dict[str, Any]]:\n    """Resolve requested markers against exact normalized feature names."""\n\n    normalized_to_indices: dict[str, list[int]] = {}\n    for index, name in enumerate(catalog.feature_names):\n        normalized_to_indices.setdefault(_normalise_token(name), []).append(index)\n\n    requested_groups: dict[str, Sequence[str]] = {**markers, **targets}\n    resolution: dict[str, Any] = {"groups": {}, "unresolved": {}}\n    selected: set[int] = set()\n    for group, aliases in requested_groups.items():\n        resolved: list[dict[str, Any]] = []\n        unresolved: list[str] = []\n        for alias in aliases:\n            candidates = normalized_to_indices.get(_normalise_token(alias), [])\n            if not candidates:\n                unresolved.append(alias)\n                continue\n            for index in candidates:\n                selected.add(index)\n                resolved.append(\n                    {\n                        "alias": alias,\n                        "index": index,\n                        "feature_id": catalog.feature_ids[index],\n                        "feature_name": catalog.feature_names[index],\n                        "feature_type": catalog.feature_types[index],\n                        "column": canonical_feature_name(\n                            catalog.feature_names[index], catalog.feature_types[index]\n                        ),\n                    }\n                )\n        resolution["groups"][group] = resolved\n        if unresolved:\n            resolution["unresolved"][group] = unresolved\n    if include_all_proteins:\n        for index, feature_type in enumerate(catalog.feature_types):\n            if _is_protein_feature_type(feature_type):\n                selected.add(index)\n    ordered = sorted(selected)\n    resolution["selected_indices"] = ordered\n    resolution["selected_count"] = len(ordered)\n    return ordered, resolution\n\n\ndef summarise_h5_features(\n    path: Path,\n    catalog: XeniumFeatureCatalog,\n    *,\n    chunk_nnz: int = 5_000_000,\n) -> pd.DataFrame:\n    """Compute totals and positive-cell fractions for all features by streaming NNZ arrays."""\n\n    import h5py\n\n    feature_count, cell_count = catalog.matrix_shape\n    sums = np.zeros(feature_count, dtype=np.float64)\n    positive_cells = np.zeros(feature_count, dtype=np.int64)\n    with h5py.File(path, "r") as handle:\n        matrix = handle["matrix"]\n        indices_ds = matrix["indices"]\n        data_ds = matrix["data"]\n        nnz = int(data_ds.shape[0])\n        for start in range(0, nnz, chunk_nnz):\n            stop = min(start + chunk_nnz, nnz)\n            indices = np.asarray(indices_ds[start:stop], dtype=np.int64)\n            values = np.asarray(data_ds[start:stop], dtype=np.float64)\n            sums += np.bincount(indices, weights=values, minlength=feature_count)\n            positive_cells += np.bincount(indices, minlength=feature_count)\n    return pd.DataFrame(\n        {\n            "feature_index": np.arange(feature_count, dtype=np.int64),\n            "feature_id": catalog.feature_ids,\n            "feature_name": catalog.feature_names,\n            "feature_type": catalog.feature_types,\n            "total_signal": sums,\n            "mean_signal_per_cell": sums / max(cell_count, 1),\n            "positive_cells": positive_cells,\n            "positive_fraction": positive_cells / max(cell_count, 1),\n        }\n    )\n\n\ndef extract_selected_h5_features(\n    path: Path,\n    catalog: XeniumFeatureCatalog,\n    selected_indices: Sequence[int],\n    *,\n    chunk_cells: int = 20_000,\n) -> pd.DataFrame:\n    """Read selected features from CSC HDF5 without materializing the full matrix."""\n\n    import h5py\n\n    feature_count, cell_count = catalog.matrix_shape\n    selected = np.asarray(sorted(set(int(value) for value in selected_indices)), dtype=np.int64)\n    if selected.size == 0:\n        raise ValueError("No selected features")\n    if selected.min() < 0 or selected.max() >= feature_count:\n        raise IndexError("Selected feature index is outside matrix bounds")\n    lookup = np.full(feature_count, -1, dtype=np.int64)\n    lookup[selected] = np.arange(selected.size, dtype=np.int64)\n    output = np.zeros((cell_count, selected.size), dtype=np.float32)\n    with h5py.File(path, "r") as handle:\n        matrix = handle["matrix"]\n        indptr_ds = matrix["indptr"]\n        indices_ds = matrix["indices"]\n        data_ds = matrix["data"]\n        for cell_start in range(0, cell_count, chunk_cells):\n            cell_stop = min(cell_start + chunk_cells, cell_count)\n            indptr = np.asarray(indptr_ds[cell_start : cell_stop + 1], dtype=np.int64)\n            nnz_start = int(indptr[0])\n            nnz_stop = int(indptr[-1])\n            local_indptr = indptr - nnz_start\n            indices = np.asarray(indices_ds[nnz_start:nnz_stop], dtype=np.int64)\n            values = np.asarray(data_ds[nnz_start:nnz_stop], dtype=np.float32)\n            local_cells = np.repeat(\n                np.arange(cell_stop - cell_start, dtype=np.int64), np.diff(local_indptr)\n            )\n            mapped = lookup[indices]\n            keep = mapped >= 0\n            block = output[cell_start:cell_stop]\n            np.add.at(block, (local_cells[keep], mapped[keep]), values[keep])\n    columns = [\n        canonical_feature_name(catalog.feature_names[index], catalog.feature_types[index])\n        for index in selected\n    ]\n    if len(columns) != len(set(columns)):\n        raise ValueError("Canonical selected-feature columns are not unique")\n    table = pd.DataFrame(output, columns=columns)\n    table.insert(0, "cell_id", list(catalog.barcodes))\n    return table\n\n\ndef robust_scale(\n    values: Sequence[float],\n    *,\n    lower_quantile: float = 0.05,\n    upper_quantile: float = 0.99,\n) -> FloatArray:\n    """Map non-negative signal to [0, 1] with log1p and robust quantiles."""\n\n    array = np.maximum(np.asarray(values, dtype=np.float64), 0.0)\n    transformed = np.log1p(array)\n    low = float(np.quantile(transformed, lower_quantile))\n    high = float(np.quantile(transformed, upper_quantile))\n    if not math.isfinite(low) or not math.isfinite(high) or high <= low:\n        return np.zeros_like(transformed)\n    return np.clip((transformed - low) / (high - low), 0.0, 1.0)\n\n\ndef otsu_threshold(values: Sequence[float], *, bins: int = 256) -> float:\n    """Compute Otsu\'s one-dimensional threshold without an image dependency."""\n\n    array = np.asarray(values, dtype=np.float64)\n    array = array[np.isfinite(array)]\n    if array.size == 0:\n        return 0.5\n    low, high = float(array.min()), float(array.max())\n    if high <= low:\n        return high\n    hist, edges = np.histogram(array, bins=bins, range=(low, high))\n    hist = hist.astype(np.float64)\n    probabilities = hist / max(hist.sum(), 1.0)\n    centres = (edges[:-1] + edges[1:]) / 2.0\n    cumulative_weight = np.cumsum(probabilities)\n    cumulative_mean = np.cumsum(probabilities * centres)\n    total_mean = cumulative_mean[-1]\n    denominator = cumulative_weight * (1.0 - cumulative_weight)\n    between = np.zeros_like(denominator)\n    valid = denominator > 0\n    between[valid] = (\n        total_mean * cumulative_weight[valid] - cumulative_mean[valid]\n    ) ** 2 / denominator[valid]\n    return float(centres[int(np.argmax(between))])\n\n\ndef _columns_for_resolution(\n    resolution: Mapping[str, Any], group: str, available_columns: Iterable[str]\n) -> list[str]:\n    available = set(available_columns)\n    return [\n        str(item["column"])\n        for item in resolution["groups"].get(group, [])\n        if str(item["column"]) in available\n    ]\n\n\ndef group_score(table: pd.DataFrame, columns: Sequence[str]) -> FloatArray:\n    """Compute a conservative group score as the mean of per-feature robust scales."""\n\n    if not columns:\n        return np.zeros(len(table), dtype=np.float64)\n    scaled = np.column_stack([robust_scale(table[column].to_numpy()) for column in columns])\n    return np.asarray(np.mean(scaled, axis=1), dtype=np.float64)\n\n\ndef threshold_aligned_scale(values: Sequence[float], threshold: float) -> FloatArray:\n    """Scale values so the declared positivity threshold maps exactly to 0.5."""\n\n    array = np.clip(np.asarray(values, dtype=np.float64), 0.0, 1.0)\n    threshold = float(np.clip(threshold, 1.0e-6, 1.0 - 1.0e-6))\n    below = 0.5 * array / threshold\n    above = 0.5 + 0.5 * (array - threshold) / (1.0 - threshold)\n    return np.where(array <= threshold, below, above)\n\n\ndef local_mean_scores(\n    coordinates: NDArray[np.float64],\n    values: FloatArray,\n    *,\n    neighbours: int = 24,\n    maximum_distance_um: float = 100.0,\n    chunk_size: int = 50_000,\n) -> FloatArray:\n    """Average a cell score over nearby cells using a bounded nearest-neighbour query."""\n\n    if coordinates.ndim != 2 or coordinates.shape[1] != 2:\n        raise ValueError("coordinates must have shape (n_cells, 2)")\n    if len(values) != len(coordinates):\n        raise ValueError("values and coordinates differ in length")\n    count = len(coordinates)\n    if count == 0:\n        return np.array([], dtype=np.float64)\n    k = min(max(neighbours, 1), count)\n    tree = cKDTree(coordinates)\n    output = np.empty(count, dtype=np.float64)\n    for start in range(0, count, chunk_size):\n        stop = min(start + chunk_size, count)\n        distances, indices = tree.query(coordinates[start:stop], k=k, workers=-1)\n        if k == 1:\n            distances = distances[:, None]\n            indices = indices[:, None]\n        valid = distances <= maximum_distance_um\n        weights = valid.astype(np.float64)\n        denominator = np.maximum(weights.sum(axis=1), 1.0)\n        output[start:stop] = (values[indices] * weights).sum(axis=1) / denominator\n    return output\n\n\ndef nearest_vessel_geometry(\n    coordinates: NDArray[np.float64],\n    vessel_positive: NDArray[np.bool_],\n    *,\n    density_radius_um: float = 100.0,\n    density_neighbours: int = 32,\n    chunk_size: int = 50_000,\n) -> tuple[FloatArray, FloatArray]:\n    """Compute nearest endothelial-cell distance and a bounded local vessel-density proxy."""\n\n    vessel_coordinates = coordinates[vessel_positive]\n    if len(vessel_coordinates) == 0:\n        return (\n            np.full(len(coordinates), np.nan, dtype=np.float64),\n            np.zeros(len(coordinates), dtype=np.float64),\n        )\n    tree = cKDTree(vessel_coordinates)\n    distances = np.empty(len(coordinates), dtype=np.float64)\n    local_density = np.empty(len(coordinates), dtype=np.float64)\n    k = min(max(density_neighbours, 1), len(vessel_coordinates))\n    for start in range(0, len(coordinates), chunk_size):\n        stop = min(start + chunk_size, len(coordinates))\n        nearest, _ = tree.query(coordinates[start:stop], k=1, workers=-1)\n        distances[start:stop] = nearest\n        neighbourhood, _ = tree.query(coordinates[start:stop], k=k, workers=-1)\n        if k == 1:\n            neighbourhood = neighbourhood[:, None]\n        local_density[start:stop] = np.mean(neighbourhood <= density_radius_um, axis=1)\n    return distances, local_density\n\n\ndef score_cells(\n    cells: pd.DataFrame,\n    expression: pd.DataFrame,\n    resolution: Mapping[str, Any],\n    *,\n    local_neighbours: int = 24,\n    local_radius_um: float = 100.0,\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    """Create transparent marker scores, vessel geometry and target signals."""\n\n    merged = cells.merge(expression, on="cell_id", how="left", validate="one_to_one")\n    feature_columns = [column for column in expression.columns if column != "cell_id"]\n    merged[feature_columns] = merged[feature_columns].fillna(0.0)\n    coordinates = merged[["x_um", "y_um"]].to_numpy(dtype=np.float64)\n\n    score_groups = (\n        "endothelial",\n        "pericyte_smooth_muscle",\n        "caf",\n        "ecm",\n        "epithelial_malignant",\n        "immune",\n    )\n    group_columns: dict[str, list[str]] = {}\n    for group in score_groups:\n        columns = _columns_for_resolution(resolution, group, merged.columns)\n        group_columns[group] = columns\n        merged[f"{group}_score"] = group_score(merged, columns)\n\n    vessel_raw = np.maximum(\n        merged["endothelial_score"].to_numpy(dtype=np.float64),\n        0.5 * merged["pericyte_smooth_muscle_score"].to_numpy(dtype=np.float64),\n    )\n    vessel_threshold = otsu_threshold(vessel_raw)\n    # Guard against pathological Otsu splits by requiring a nontrivial high-confidence tail.\n    vessel_threshold = float(np.clip(vessel_threshold, 0.25, 0.85))\n    vessel_signal = threshold_aligned_scale(vessel_raw, vessel_threshold)\n    vessel_positive = vessel_signal >= 0.5\n    if vessel_positive.mean() < 0.001:\n        fallback = float(np.quantile(vessel_raw, 0.99))\n        vessel_signal = threshold_aligned_scale(vessel_raw, max(fallback, 1.0e-6))\n        vessel_positive = vessel_signal >= 0.5\n        vessel_threshold = fallback\n\n    local_caf = local_mean_scores(\n        coordinates,\n        merged["caf_score"].to_numpy(dtype=np.float64),\n        neighbours=local_neighbours,\n        maximum_distance_um=local_radius_um,\n    )\n    local_ecm = local_mean_scores(\n        coordinates,\n        merged["ecm_score"].to_numpy(dtype=np.float64),\n        neighbours=local_neighbours,\n        maximum_distance_um=local_radius_um,\n    )\n    vessel_distance, local_vessel_density = nearest_vessel_geometry(\n        coordinates, vessel_positive\n    )\n    merged["vessel_signal"] = vessel_signal\n    merged["vessel_positive"] = vessel_positive\n    merged["distance_to_vessel_um"] = vessel_distance\n    merged["local_vessel_density"] = local_vessel_density\n    merged["local_caf_score"] = np.clip(local_caf, 0.0, 1.0)\n    merged["local_ecm_score"] = np.clip(local_ecm, 0.0, 1.0)\n\n    malignant_raw = merged["epithelial_malignant_score"].to_numpy(dtype=np.float64)\n    malignant_threshold = float(np.clip(otsu_threshold(malignant_raw), 0.20, 0.85))\n    malignant = malignant_raw >= malignant_threshold\n    merged["cell_is_malignant_proxy"] = malignant\n    if np.any(malignant):\n        malignant_tree = cKDTree(coordinates[malignant])\n        distance_to_malignant, _ = malignant_tree.query(coordinates, k=1, workers=-1)\n        merged["distance_to_malignant_proxy_um"] = distance_to_malignant\n        merged["in_molecular_tumour_neighbourhood"] = distance_to_malignant <= 150.0\n    else:\n        merged["distance_to_malignant_proxy_um"] = np.nan\n        merged["in_molecular_tumour_neighbourhood"] = False\n\n    target_diagnostics: dict[str, Any] = {}\n    for target in DEFAULT_TARGET_ALIASES:\n        columns = _columns_for_resolution(resolution, target, merged.columns)\n        if not columns:\n            continue\n        # Prefer direct protein intensity; use RNA only as a separately identified fallback.\n        protein_columns = [column for column in columns if column.startswith("protein__")]\n        active_columns = protein_columns or columns\n        target_raw = group_score(merged, active_columns)\n        threshold = float(np.clip(otsu_threshold(target_raw), 0.10, 0.90))\n        signal = threshold_aligned_scale(target_raw, threshold)\n        safe_name = target.replace("-", "_")\n        merged[f"target__{safe_name}__signal"] = signal\n        merged[f"target__{safe_name}__positive"] = signal >= 0.5\n        target_diagnostics[target] = {\n            "columns": active_columns,\n            "measurement": "protein_intensity" if protein_columns else "rna_proxy",\n            "raw_threshold": threshold,\n            "positive_cells": int(np.sum(signal >= 0.5)),\n            "positive_fraction": float(np.mean(signal >= 0.5)),\n        }\n\n    diagnostics = {\n        "group_columns": group_columns,\n        "vessel_threshold": vessel_threshold,\n        "vessel_positive_cells": int(vessel_positive.sum()),\n        "vessel_positive_fraction": float(vessel_positive.mean()),\n        "malignant_proxy_threshold": malignant_threshold,\n        "malignant_proxy_cells": int(malignant.sum()),\n        "targets": target_diagnostics,\n        "warnings": [\n            "Endothelial-cell presence is not a measurement of vessel perfusion.",\n            "Xenium protein signal is scaled mean fluorescence intensity, "\n            "not antigen molecules per cell.",\n            "The molecular tumour neighbourhood is a fallback proxy until "\n            "pathology alignment is verified.",\n        ],\n    }\n    return merged, diagnostics\n\n\ndef load_affine_matrix(path: Path) -> NDArray[np.float64]:\n    """Read the 3x3 affine transformation supplied by Xenium Explorer."""\n\n    matrix = np.loadtxt(path, delimiter=",")\n    if matrix.shape != (3, 3):\n        raise ValueError(f"Expected a 3x3 alignment matrix, observed {matrix.shape}")\n    if not np.allclose(matrix[2], np.array([0.0, 0.0, 1.0]), atol=1.0e-8):\n        raise ValueError("Alignment matrix final row is not [0, 0, 1]")\n    return np.asarray(matrix, dtype=np.float64)\n\n\ndef geojson_vertices(path: Path) -> NDArray[np.float64]:\n    """Flatten Polygon and MultiPolygon vertices from a GeoJSON file."""\n\n    document = json.loads(path.read_text(encoding="utf-8"))\n    vertices: list[tuple[float, float]] = []\n    for feature in document.get("features", []):\n        geometry = feature.get("geometry", {})\n        geometry_type = geometry.get("type")\n        coordinates = geometry.get("coordinates", [])\n        polygons = coordinates if geometry_type == "MultiPolygon" else [coordinates]\n        if geometry_type not in {"Polygon", "MultiPolygon"}:\n            continue\n        for polygon in polygons:\n            for ring in polygon:\n                for coordinate in ring:\n                    vertices.append((float(coordinate[0]), float(coordinate[1])))\n    if not vertices:\n        raise ValueError("GeoJSON contains no polygon vertices")\n    return np.asarray(vertices, dtype=np.float64)\n\n\ndef _bbox_score(\n    transformed: NDArray[np.float64],\n    cell_bbox: tuple[float, float, float, float],\n) -> tuple[float, float]:\n    x_min, x_max, y_min, y_max = cell_bbox\n    margin_x = max((x_max - x_min) * 0.10, 100.0)\n    margin_y = max((y_max - y_min) * 0.10, 100.0)\n    inside = (\n        (transformed[:, 0] >= x_min - margin_x)\n        & (transformed[:, 0] <= x_max + margin_x)\n        & (transformed[:, 1] >= y_min - margin_y)\n        & (transformed[:, 1] <= y_max + margin_y)\n    )\n    fraction_inside = float(np.mean(inside))\n    tx_min, ty_min = transformed.min(axis=0)\n    tx_max, ty_max = transformed.max(axis=0)\n    cell_width = max(x_max - x_min, 1.0)\n    cell_height = max(y_max - y_min, 1.0)\n    width_ratio = max((tx_max - tx_min) / cell_width, 1.0e-9)\n    height_ratio = max((ty_max - ty_min) / cell_height, 1.0e-9)\n    shape_penalty = abs(math.log(width_ratio)) + abs(math.log(height_ratio))\n    centre_distance = math.hypot(\n        ((tx_min + tx_max) - (x_min + x_max)) / (2.0 * cell_width),\n        ((ty_min + ty_max) - (y_min + y_max)) / (2.0 * cell_height),\n    )\n    score = fraction_inside - 0.15 * shape_penalty - 0.10 * centre_distance\n    return score, fraction_inside\n\n\ndef infer_annotation_transform(\n    vertices: NDArray[np.float64],\n    cells: pd.DataFrame,\n    affine: NDArray[np.float64],\n    *,\n    candidate_pixel_sizes_um: Sequence[float] = (1.0, 0.2125, 0.425, 0.5),\n) -> tuple[AlignmentCandidate | None, list[AlignmentCandidate]]:\n    """Evaluate transform direction and pixel scaling; abstain when ambiguous."""\n\n    homogeneous = np.column_stack([vertices, np.ones(len(vertices), dtype=np.float64)])\n    cell_bbox = (\n        float(cells["x_um"].min()),\n        float(cells["x_um"].max()),\n        float(cells["y_um"].min()),\n        float(cells["y_um"].max()),\n    )\n    transformations = {\n        "identity": np.eye(3, dtype=np.float64),\n        "affine": affine,\n        "inverse_affine": np.linalg.inv(affine),\n    }\n    candidates: list[AlignmentCandidate] = []\n    seen: set[tuple[float, ...]] = set()\n    for transform_name, matrix in transformations.items():\n        transformed_pixels = (matrix @ homogeneous.T).T[:, :2]\n        for scale in candidate_pixel_sizes_um:\n            signature = tuple(np.round((matrix * float(scale)).ravel(), 10).tolist())\n            if signature in seen:\n                continue\n            seen.add(signature)\n            transformed = transformed_pixels * np.array([scale, scale])\n            score, fraction_inside = _bbox_score(transformed, cell_bbox)\n            candidates.append(\n                AlignmentCandidate(\n                    name=f"{transform_name}_scale_{scale:g}",\n                    matrix=matrix,\n                    scale_x=float(scale),\n                    scale_y=float(scale),\n                    score=score,\n                    fraction_inside=fraction_inside,\n                )\n            )\n    candidates.sort(key=lambda candidate: candidate.score, reverse=True)\n    best = candidates[0]\n    runner_up = candidates[1]\n    if best.fraction_inside < 0.70 or best.score - runner_up.score < 0.05:\n        return None, candidates\n    return best, candidates\n\n\ndef transform_geojson(\n    input_path: Path,\n    output_path: Path,\n    candidate: AlignmentCandidate,\n) -> None:\n    """Transform a GeoJSON annotation layer into inferred Xenium micron coordinates."""\n\n    document = json.loads(input_path.read_text(encoding="utf-8"))\n\n    def transform_coordinate(coordinate: Sequence[float]) -> list[float]:\n        vector = np.array([float(coordinate[0]), float(coordinate[1]), 1.0])\n        result = candidate.matrix @ vector\n        return [float(result[0] * candidate.scale_x), float(result[1] * candidate.scale_y)]\n\n    for feature in document.get("features", []):\n        geometry = feature.get("geometry", {})\n        geometry_type = geometry.get("type")\n        coordinates = geometry.get("coordinates", [])\n        if geometry_type == "Polygon":\n            geometry["coordinates"] = [\n                [transform_coordinate(coordinate) for coordinate in ring]\n                for ring in coordinates\n            ]\n        elif geometry_type == "MultiPolygon":\n            geometry["coordinates"] = [\n                [\n                    [transform_coordinate(coordinate) for coordinate in ring]\n                    for ring in polygon\n                ]\n                for polygon in coordinates\n            ]\n    document.setdefault("reach_gap", {})["inferred_transform"] = {\n        "name": candidate.name,\n        "matrix": candidate.matrix.tolist(),\n        "scale_x": candidate.scale_x,\n        "scale_y": candidate.scale_y,\n        "score": candidate.score,\n        "fraction_inside": candidate.fraction_inside,\n    }\n    output_path.write_text(json.dumps(document), encoding="utf-8")\n\n\ndef _annotation_names(document: Mapping[str, Any]) -> list[str]:\n    names: list[str] = []\n    for feature in document.get("features", []):\n        properties = feature.get("properties", {})\n        classification = properties.get("classification") or {}\n        name = properties.get("name") or classification.get("name")\n        names.append(str(name or "UNLABELLED"))\n    return names\n\n\ndef assign_pathology_regions(cells: pd.DataFrame, geojson_path: Path) -> pd.DataFrame:\n    """Assign polygon labels to cell centroids with explicit overlap handling."""\n\n    from shapely.geometry import Point, shape\n    from shapely.strtree import STRtree\n\n    document = json.loads(geojson_path.read_text(encoding="utf-8"))\n    geometries = [shape(feature["geometry"]) for feature in document.get("features", [])]\n    names = _annotation_names(document)\n    if not geometries:\n        output = cells.copy()\n        output["pathology_region"] = "UNANNOTATED"\n        return output\n    tree = STRtree(geometries)\n    try:\n        from shapely import points\n\n        point_array = points(\n            cells["x_um"].to_numpy(dtype=np.float64),\n            cells["y_um"].to_numpy(dtype=np.float64),\n        )\n        pairs = tree.query(point_array, predicate="intersects")\n        matched_by_cell: dict[int, set[str]] = {}\n        if pairs.size:\n            for cell_index, geometry_index in pairs.T:\n                matched_by_cell.setdefault(int(cell_index), set()).add(\n                    names[int(geometry_index)]\n                )\n        labels = [\n            "|".join(sorted(matched_by_cell[index]))\n            if index in matched_by_cell\n            else "UNANNOTATED"\n            for index in range(len(cells))\n        ]\n    except (ImportError, AttributeError):\n        labels = []\n        for x, y in cells[["x_um", "y_um"]].itertuples(index=False, name=None):\n            point = Point(float(x), float(y))\n            candidate_indices = tree.query(point, predicate="intersects")\n            if len(candidate_indices) == 0:\n                labels.append("UNANNOTATED")\n            else:\n                matched = sorted({names[int(index)] for index in candidate_indices})\n                labels.append("|".join(matched))\n    output = cells.copy()\n    output["pathology_region"] = labels\n    return output\n\n\ndef write_partitioned_table(\n    table: pd.DataFrame,\n    output_dir: Path,\n    stem: str,\n    *,\n    rows_per_part: int = 100_000,\n) -> list[str]:\n    """Write bounded-size table parts, preferring Zstandard-compressed Parquet."""\n\n    if rows_per_part < 1:\n        raise ValueError("rows_per_part must be positive")\n    output_dir.mkdir(parents=True, exist_ok=True)\n    paths: list[str] = []\n    try:\n        import pyarrow  # noqa: F401\n\n        suffix = "parquet"\n        writer = lambda frame, path: frame.to_parquet(  # noqa: E731\n            path, index=False, compression="zstd"\n        )\n    except ImportError:\n        suffix = "csv.gz"\n        writer = lambda frame, path: frame.to_csv(  # noqa: E731\n            path, index=False, compression="gzip"\n        )\n    for part, start in enumerate(range(0, len(table), rows_per_part)):\n        stop = min(start + rows_per_part, len(table))\n        path = output_dir / f"{stem}.part{part:04d}.{suffix}"\n        writer(table.iloc[start:stop], path)\n        paths.append(str(path))\n    if not paths:\n        path = output_dir / f"{stem}.part0000.{suffix}"\n        writer(table, path)\n        paths.append(str(path))\n    return paths\n\n\ndef read_partitioned_table(paths: Sequence[Path]) -> pd.DataFrame:\n    """Read table parts produced by :func:`write_partitioned_table`."""\n\n    frames: list[pd.DataFrame] = []\n    for path in sorted(paths):\n        if path.suffix == ".parquet":\n            frames.append(pd.read_parquet(path))\n        elif path.name.endswith(".csv.gz"):\n            frames.append(pd.read_csv(path, compression="gzip"))\n        else:\n            frames.append(pd.read_csv(path))\n    if not frames:\n        raise FileNotFoundError("No table parts were supplied")\n    return pd.concat(frames, ignore_index=True)\n\n\ndef build_target_tables(\n    scored: pd.DataFrame,\n    output_dir: Path,\n    diagnostics: Mapping[str, Any],\n) -> dict[str, list[str]]:\n    """Write generic reach-gap input tables for every resolved target."""\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    outputs: dict[str, list[str]] = {}\n    if "pathology_region" in scored:\n        pathology_tumour = scored["pathology_region"].astype(str).str.contains(\n            "Tumor", case=False, regex=False\n        )\n        in_tumour_region = pathology_tumour.to_numpy(dtype=np.bool_)\n        region_definition = "pathology_tumour_polygon"\n    else:\n        in_tumour_region = scored["in_molecular_tumour_neighbourhood"].to_numpy(dtype=np.bool_)\n        region_definition = "molecular_tumour_neighbourhood_150um"\n\n    for target, target_info in diagnostics.get("targets", {}).items():\n        safe_name = target.replace("-", "_")\n        signal_column = f"target__{safe_name}__signal"\n        if signal_column not in scored:\n            continue\n        table = pd.DataFrame(\n            {\n                "cell_id": scored["cell_id"].astype(str),\n                "x_um": scored["x_um"].astype(float),\n                "y_um": scored["y_um"].astype(float),\n                # Historical schema name: this means \'cell lies in tumour region\', not malignancy.\n                "is_tumour": in_tumour_region,\n                "target_signal": scored[signal_column].astype(float),\n                "vessel_signal": scored["vessel_signal"].astype(float),\n                "ecm_score": scored["local_ecm_score"].astype(float),\n                "caf_score": scored["local_caf_score"].astype(float),\n                "cell_is_malignant_proxy": scored["cell_is_malignant_proxy"].astype(bool),\n                "distance_to_vessel_um": scored["distance_to_vessel_um"].astype(float),\n                "target_measurement": str(target_info["measurement"]),\n                "tumour_region_definition": region_definition,\n            }\n        )\n        outputs[target] = write_partitioned_table(\n            table, output_dir, f"reach_gap_cells_{safe_name}"\n        )\n    return outputs\n\n\ndef write_manifest(\n    output_path: Path,\n    *,\n    source_files: Sequence[Path],\n    zip_inventory: pd.DataFrame,\n    extracted: Mapping[str, str],\n    diagnostics: Mapping[str, Any],\n    alignment: AlignmentCandidate | None,\n) -> None:\n    """Write an auditable preparation manifest with explicit unsupported claims."""\n\n    manifest = {\n        "schema_version": "1.0",\n        "dataset": (\n            "Xenium In Situ Gene and Protein Expression data for FFPE Human "\n            "Renal Cell Carcinoma"\n        ),\n        "platform": "Xenium Onboard Analysis 4.0",\n        "licence": "CC BY 4.0",\n        "source_files": [\n            {\n                "path": str(path),\n                "size_bytes": path.stat().st_size,\n                "sha256": (\n                    sha256_file(path)\n                    if path.stat().st_size < 1_000_000_000\n                    else "NOT_COMPUTED_LARGE_FILE"\n                ),\n            }\n            for path in source_files\n            if path.exists()\n        ],\n        "zip_members": int(len(zip_inventory)),\n        "zip_uncompressed_bytes": int(zip_inventory["uncompressed_bytes"].sum()),\n        "extracted": dict(extracted),\n        "diagnostics": diagnostics,\n        "annotation_alignment": None\n        if alignment is None\n        else {\n            "name": alignment.name,\n            "matrix": alignment.matrix.tolist(),\n            "scale_x": alignment.scale_x,\n            "scale_y": alignment.scale_y,\n            "score": alignment.score,\n            "fraction_inside": alignment.fraction_inside,\n        },\n        "calibration": {\n            "antigen_molecules_per_cell": "NOT_COMPUTED",\n            "antigen_nM_per_signal": None,\n            "reason": (\n                "Xenium protein values are scaled mean fluorescence intensity, "\n                "not absolute surface density."\n            ),\n        },\n        "perfusion": {\n            "measured": False,\n            "reason": "CD31/endothelial presence does not establish functional perfusion.",\n        },\n        "permitted_claims": [\n            "Cell-level RNA and protein signals were prepared from the complete "\n            "Xenium cell-feature matrix.",\n            "Distances are measured to high-confidence endothelial-cell proxies "\n            "in the same section.",\n            "Target and barrier scores are relative, transparent marker-derived quantities.",\n        ],\n        "unsupported_claims": [\n            "Absolute antibody penetration or receptor occupancy.",\n            "Clinical efficacy prediction.",\n            "Perfused-vessel identification.",\n            "Absolute surface-antigen density.",\n        ],\n    }\n    output_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n\n\ndef prepare_rcc_xenium(\n    *,\n    raw_dir: Path,\n    output_dir: Path,\n    verify_large_md5: bool = True,\n    force: bool = False,\n) -> dict[str, Any]:\n    """Run the complete in-Drive preparation pipeline with resumable checkpoints."""\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    verification_path = output_dir / "download_verification.json"\n    if verification_path.exists() and not force:\n        verify_report = json.loads(verification_path.read_text(encoding="utf-8"))\n        cached_sizes_valid = all(\n            (raw_dir / name).exists()\n            and (raw_dir / name).stat().st_size == int(specification["size"])\n            for name, specification in EXPECTED_RCC_FILES.items()\n        )\n        if not (\n            verify_report.get("all_present")\n            and verify_report.get("all_verified")\n            and cached_sizes_valid\n        ):\n            verify_report = verify_expected_files(\n                raw_dir, verify_large_md5=verify_large_md5\n            )\n    else:\n        verify_report = verify_expected_files(raw_dir, verify_large_md5=verify_large_md5)\n    verification_path.write_text(json.dumps(verify_report, indent=2), encoding="utf-8")\n    if not verify_report["all_present"]:\n        missing = [name for name, entry in verify_report["files"].items() if not entry["present"]]\n        raise FileNotFoundError(f"Missing downloaded files: {missing}")\n    if not verify_report["all_verified"]:\n        raise ValueError("At least one downloaded file failed size or MD5 verification")\n\n    zip_path = raw_dir / "Xenium_V1_Human_Kidney_FFPE_Protein_updated_outs.zip"\n    he_path = raw_dir / "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_image.ome.tif"\n    alignment_path = raw_dir / "Xenium_V1_Human_Kidney_FFPE_Protein_updated_he_imagealignment.csv"\n    annotation_path = raw_dir / "Xenium_V1_Human_Kidney_FFPE_Protein_updated_annotation.geojson"\n\n    inventory_path = output_dir / "zip_inventory.csv"\n    if inventory_path.exists() and not force:\n        inventory = pd.read_csv(inventory_path)\n    else:\n        inventory = inspect_zip(zip_path)\n        inventory.to_csv(inventory_path, index=False)\n\n    extracted_manifest_path = output_dir / "extracted_members.json"\n    extracted_dir = output_dir / "extracted"\n    extracted: dict[str, str]\n    if extracted_manifest_path.exists() and not force:\n        cached = json.loads(extracted_manifest_path.read_text(encoding="utf-8"))\n        cached_paths_valid = cached and all(\n            Path(path).exists() and Path(path).stat().st_size > 0\n            for path in cached.values()\n        )\n        if cached_paths_valid:\n            extracted = {str(key): str(value) for key, value in cached.items()}\n        else:\n            members = select_essential_members(inventory)\n            extracted = extract_members(zip_path, members, extracted_dir)\n    else:\n        members = select_essential_members(inventory)\n        extracted = extract_members(zip_path, members, extracted_dir)\n    extracted_manifest_path.write_text(json.dumps(extracted, indent=2), encoding="utf-8")\n\n    cells_path = find_extracted(extracted_dir, "cells.parquet") or find_extracted(\n        extracted_dir, "cells.csv.gz"\n    )\n    h5_path = find_extracted(extracted_dir, "cell_feature_matrix.h5")\n    if cells_path is None or h5_path is None:\n        raise FileNotFoundError("Essential cells or HDF5 matrix file was not extracted")\n    cells = read_cells(cells_path)\n    catalog = read_10x_h5_catalog(h5_path)\n    validate_cell_barcode_identity(cells, catalog)\n\n    feature_summary_path = output_dir / "feature_summary.csv"\n    resolution_path = output_dir / "marker_resolution.json"\n    selected_parts_manifest = output_dir / "selected_expression_parts.json"\n    selected_expression: pd.DataFrame\n    if (\n        feature_summary_path.exists()\n        and resolution_path.exists()\n        and selected_parts_manifest.exists()\n        and not force\n    ):\n        selected_part_paths = [\n            Path(value)\n            for value in json.loads(selected_parts_manifest.read_text(encoding="utf-8"))\n        ]\n        if selected_part_paths and all(path.exists() for path in selected_part_paths):\n            resolution = json.loads(resolution_path.read_text(encoding="utf-8"))\n            selected_indices = [int(value) for value in resolution["selected_indices"]]\n            selected_expression = read_partitioned_table(selected_part_paths)\n        else:\n            selected_indices, resolution = resolve_feature_indices(catalog)\n            selected_expression = extract_selected_h5_features(\n                h5_path, catalog, selected_indices\n            )\n            selected_part_paths = [\n                Path(value)\n                for value in write_partitioned_table(\n                    selected_expression, output_dir / "tables", "selected_expression"\n                )\n            ]\n    else:\n        feature_summary = summarise_h5_features(h5_path, catalog)\n        feature_summary.to_csv(feature_summary_path, index=False)\n        selected_indices, resolution = resolve_feature_indices(catalog)\n        resolution_path.write_text(json.dumps(resolution, indent=2), encoding="utf-8")\n        selected_expression = extract_selected_h5_features(h5_path, catalog, selected_indices)\n        selected_part_paths = [\n            Path(value)\n            for value in write_partitioned_table(\n                selected_expression, output_dir / "tables", "selected_expression"\n            )\n        ]\n    if not feature_summary_path.exists():\n        summarise_h5_features(h5_path, catalog).to_csv(feature_summary_path, index=False)\n    if not resolution_path.exists():\n        resolution_path.write_text(json.dumps(resolution, indent=2), encoding="utf-8")\n    selected_expression_parts = [str(path) for path in selected_part_paths]\n    selected_parts_manifest.write_text(\n        json.dumps(selected_expression_parts, indent=2), encoding="utf-8"\n    )\n\n    diagnostics_path = output_dir / "cell_scoring_diagnostics.json"\n    scored_parts_manifest = output_dir / "scored_cell_parts.json"\n    if diagnostics_path.exists() and scored_parts_manifest.exists() and not force:\n        scored_part_paths = [\n            Path(value)\n            for value in json.loads(scored_parts_manifest.read_text(encoding="utf-8"))\n        ]\n        if scored_part_paths and all(path.exists() for path in scored_part_paths):\n            diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))\n            scored = read_partitioned_table(scored_part_paths)\n        else:\n            scored, diagnostics = score_cells(cells, selected_expression, resolution)\n            scored_part_paths = []\n    else:\n        scored, diagnostics = score_cells(cells, selected_expression, resolution)\n        scored_part_paths = []\n\n    alignment_candidates_path = output_dir / "annotation_alignment_candidates.json"\n    transformed_path = output_dir / "pathology_annotations_xenium.geojson"\n    best_alignment: AlignmentCandidate | None = None\n    if transformed_path.exists() and alignment_candidates_path.exists() and not force:\n        candidate_records = json.loads(alignment_candidates_path.read_text(encoding="utf-8"))\n        accepted = [record for record in candidate_records if record.get("accepted")]\n        if accepted:\n            record = accepted[0]\n            best_alignment = AlignmentCandidate(\n                name=str(record["name"]),\n                matrix=np.asarray(record["matrix"], dtype=np.float64),\n                scale_x=float(record["scale_x"]),\n                scale_y=float(record["scale_y"]),\n                score=float(record["score"]),\n                fraction_inside=float(record["fraction_inside"]),\n            )\n            if "pathology_region" not in scored.columns:\n                scored = assign_pathology_regions(scored, transformed_path)\n    else:\n        alignment = load_affine_matrix(alignment_path)\n        vertices = geojson_vertices(annotation_path)\n        best_alignment, candidates = infer_annotation_transform(vertices, cells, alignment)\n        candidate_records = [\n            {\n                "name": candidate.name,\n                "matrix": candidate.matrix.tolist(),\n                "scale_x": candidate.scale_x,\n                "scale_y": candidate.scale_y,\n                "score": candidate.score,\n                "fraction_inside": candidate.fraction_inside,\n                "accepted": best_alignment is not None and candidate.name == best_alignment.name,\n            }\n            for candidate in candidates\n        ]\n        alignment_candidates_path.write_text(\n            json.dumps(candidate_records, indent=2), encoding="utf-8"\n        )\n        if best_alignment is not None:\n            transform_geojson(annotation_path, transformed_path, best_alignment)\n            scored = assign_pathology_regions(scored, transformed_path)\n        else:\n            diagnostics.setdefault("warnings", []).append(\n                "Pathology alignment was ambiguous; tumour regions use the molecular "\n                "150 µm neighbourhood proxy."\n            )\n\n    # Re-write scored parts whenever pathology annotations were newly added or a\n    # checkpoint was incomplete.\n    if not scored_part_paths or (\n        best_alignment is not None and "pathology_region" in scored.columns\n    ):\n        scored_part_paths = [\n            Path(value)\n            for value in write_partitioned_table(\n                scored, output_dir / "tables", "cells_reach_gap"\n            )\n        ]\n    scored_parts = [str(path) for path in scored_part_paths]\n    scored_parts_manifest.write_text(json.dumps(scored_parts, indent=2), encoding="utf-8")\n    diagnostics_path.write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")\n\n    target_outputs = build_target_tables(scored, output_dir / "targets", diagnostics)\n    write_manifest(\n        output_dir / "processing_manifest.json",\n        source_files=[zip_path, he_path, alignment_path, annotation_path],\n        zip_inventory=inventory,\n        extracted=extracted,\n        diagnostics=diagnostics,\n        alignment=best_alignment,\n    )\n    result = {\n        "status": "PREPARED_WITH_ABSOLUTE_INDEX_NOT_COMPUTED",\n        "cells": len(cells),\n        "features": catalog.matrix_shape[0],\n        "selected_features": len(selected_indices),\n        "selected_expression_parts": selected_expression_parts,\n        "scored_cell_parts": scored_parts,\n        "targets": target_outputs,\n        "pathology_alignment": None if best_alignment is None else best_alignment.name,\n        "output_dir": str(output_dir),\n        "absolute_index": {\n            "status": "NOT_COMPUTED",\n            "reason": (\n                "No independent conversion from Xenium protein intensity to "\n                "surface antigen density is available."\n            ),\n        },\n    }\n    (output_dir / "run_result.json").write_text(\n        json.dumps(result, indent=2), encoding="utf-8"\n    )\n    return result\n\n'
MODULE_PATH = Path("/content/reach_gap_xenium.py")
MODULE_PATH.write_text(MODULE_SOURCE, encoding="utf-8")
print("Wrote adapter:", MODULE_PATH)


In [ ]:
import importlib
import json
import sys
import traceback

if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import reach_gap_xenium
importlib.reload(reach_gap_xenium)

try:
    result = reach_gap_xenium.prepare_rcc_xenium(
        raw_dir=RAW_DIR,
        output_dir=OUTPUT_DIR,
        verify_large_md5=VERIFY_LARGE_MD5,
        force=FORCE_RERUN,
    )
except Exception as error:
    failure = {
        "error_type": type(error).__name__,
        "message": str(error),
        "traceback": traceback.format_exc(),
        "raw_dir": str(RAW_DIR),
        "output_dir": str(OUTPUT_DIR),
    }
    (OUTPUT_DIR / "notebook_failure.json").write_text(
        json.dumps(failure, indent=2), encoding="utf-8"
    )
    print(json.dumps(failure, indent=2))
    raise
else:
    print(json.dumps(result, indent=2))


## Quality-control figures and compact handoff

The next cell reads only the partitioned prepared tables, creates diagnostic maps and writes a small ZIP containing manifests, summaries and figures. Large cell-level parts remain separately in Drive so they can be inspected or transferred individually.

In [ ]:
import json
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tifffile


def read_columns(paths, columns):
    frames = []
    for value in paths:
        path = Path(value)
        if path.suffix == ".parquet":
            available = set(pq.ParquetFile(path).schema.names)
            use = [column for column in columns if column in available]
            frames.append(pd.read_parquet(path, columns=use))
        else:
            # CSV fallback is expected only when pyarrow was unavailable during preparation.
            frame = pd.read_csv(path, compression="gzip")
            frames.append(frame[[column for column in columns if column in frame.columns]])
    return pd.concat(frames, ignore_index=True)


def write_safe_he_thumbnail(source, destination, status_path, max_dimension=4096):
    status = {"status": "NOT_COMPUTED", "source": str(source)}
    try:
        with tifffile.TiffFile(source) as tif:
            series = tif.series[0]
            levels = list(getattr(series, "levels", [series]))
            metadata = [
                {"shape": list(level.shape), "dtype": str(level.dtype)}
                for level in levels
            ]
            status["available_levels"] = metadata
            eligible = [
                level for level in levels
                if len(level.shape) >= 2 and max(level.shape[-2:]) <= max_dimension
            ]
            if not eligible:
                status["reason"] = (
                    "No pyramid level was small enough to read safely without materializing "
                    "the multi-gigabyte full-resolution image."
                )
            else:
                level = min(eligible, key=lambda item: int(np.prod(item.shape[-2:])))
                image = np.asarray(level.asarray())
                image = np.squeeze(image)
                if image.ndim == 3 and image.shape[0] in {3, 4} and image.shape[-1] not in {3, 4}:
                    image = np.moveaxis(image, 0, -1)
                plt.figure(figsize=(10, 8))
                if image.ndim == 2:
                    plt.imshow(image, cmap="gray")
                else:
                    plt.imshow(image[..., :3])
                plt.axis("off")
                plt.title("Post-Xenium H&E overview (smallest safe pyramid level)")
                plt.tight_layout()
                plt.savefig(destination, dpi=180, bbox_inches="tight")
                plt.close()
                status.update({
                    "status": "COMPUTED",
                    "output": str(destination),
                    "selected_shape": list(image.shape),
                })
    except Exception as error:
        status["reason"] = f"{type(error).__name__}: {error}"
    status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    return status


columns = [
    "cell_id", "x_um", "y_um", "distance_to_vessel_um", "vessel_positive",
    "local_caf_score", "local_ecm_score", "pathology_region",
    "in_molecular_tumour_neighbourhood",
    "target__PD_L1__signal", "target__VISTA__signal",
    "target__PD_1__signal", "target__LAG_3__signal",
]
scored = read_columns(result["scored_cell_parts"], columns)
plot_sample = scored.sample(min(120_000, len(scored)), random_state=RANDOM_SEED)
qc_dir = OUTPUT_DIR / "qc"
qc_dir.mkdir(exist_ok=True)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    plot_sample["x_um"], plot_sample["y_um"],
    c=plot_sample["distance_to_vessel_um"], s=1, alpha=0.55,
)
plt.colorbar(scatter, label="Distance to endothelial-cell proxy (µm)")
plt.gca().set_aspect("equal")
plt.gca().invert_yaxis()
plt.xlabel("X (µm)")
plt.ylabel("Y (µm)")
plt.title("Xenium RCC: nearest endothelial-cell distance")
plt.tight_layout()
plt.savefig(qc_dir / "vascular_distance_map.png", dpi=220)
plt.close()

for target, column in {
    "PD-L1": "target__PD_L1__signal",
    "VISTA": "target__VISTA__signal",
    "PD-1": "target__PD_1__signal",
    "LAG-3": "target__LAG_3__signal",
}.items():
    if column not in plot_sample.columns:
        continue
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        plot_sample["x_um"], plot_sample["y_um"],
        c=plot_sample[column], s=1, alpha=0.55,
    )
    plt.colorbar(scatter, label="Relative target signal")
    plt.gca().set_aspect("equal")
    plt.gca().invert_yaxis()
    plt.xlabel("X (µm)")
    plt.ylabel("Y (µm)")
    plt.title(f"Xenium RCC: {target} relative protein/RNA signal")
    plt.tight_layout()
    plt.savefig(qc_dir / f"target_{target.replace('-', '_')}_map.png", dpi=220)
    plt.close()

he_status = write_safe_he_thumbnail(
    RAW_DIR / EXPECTED_FILES[1],
    qc_dir / "post_xenium_he_overview.png",
    qc_dir / "he_thumbnail_status.json",
)

summary_rows = []
for column in [
    "distance_to_vessel_um", "local_caf_score", "local_ecm_score",
    "target__PD_L1__signal", "target__VISTA__signal",
    "target__PD_1__signal", "target__LAG_3__signal",
]:
    if column not in scored.columns:
        continue
    values = pd.to_numeric(scored[column], errors="coerce")
    summary_rows.append({
        "variable": column,
        "n": int(values.notna().sum()),
        "mean": float(values.mean()),
        "median": float(values.median()),
        "q05": float(values.quantile(0.05)),
        "q95": float(values.quantile(0.95)),
    })
pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / "prepared_variable_summary.csv", index=False)

if "pathology_region" in scored.columns:
    scored["pathology_region"].value_counts(dropna=False).rename_axis(
        "pathology_region"
    ).reset_index(name="cells").to_csv(
        OUTPUT_DIR / "pathology_region_counts.csv", index=False
    )

handoff = {
    "status": result["status"],
    "raw_dir": str(RAW_DIR),
    "output_dir": str(OUTPUT_DIR),
    "run_result": result,
    "he_thumbnail": he_status,
    "all_output_files": [
        {"path": str(path), "size_bytes": path.stat().st_size}
        for path in sorted(OUTPUT_DIR.rglob("*")) if path.is_file()
    ],
    "next_action": "Share the reach-gap-analysis folder with ChatGPT after this notebook finishes.",
}
(OUTPUT_DIR / "chat_handoff.json").write_text(json.dumps(handoff, indent=2), encoding="utf-8")

handoff_files = [
    path for path in OUTPUT_DIR.rglob("*")
    if path.is_file()
    and "extracted" not in path.parts
    and "tables" not in path.parts
    and "targets" not in path.parts
]
with zipfile.ZipFile(OUTPUT_DIR / "reach_gap_chat_handoff.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in handoff_files:
        if path.name == "reach_gap_chat_handoff.zip":
            continue
        archive.write(path, path.relative_to(OUTPUT_DIR))

print("Prepared cells:", len(scored))
print("Compact handoff:", OUTPUT_DIR / "reach_gap_chat_handoff.zip")
print("Full output folder:", OUTPUT_DIR)


## Completion criteria

A successful run ends with:

- `download_verification.json` reporting all files verified;
- `zip_inventory.csv` covering the complete 36.1 GB bundle;
- partitioned tables under `tables/` and `targets/`;
- `processing_manifest.json`, `run_result.json` and `chat_handoff.json`;
- QC maps under `qc/`;
- `reach_gap_chat_handoff.zip`.

The notebook is resumable. After a disconnect, reconnect and run all cells again with `FORCE_RERUN = False`; completed stages will be reused. Do not delete the `reach-gap-analysis` folder between runs.